In [2]:
import tempfile
from datetime import date
from pathlib import Path
from unittest.mock import patch

from src.ingestion.download import downloader

In [50]:
class FakeResponse:
    def __init__(self, chunks, status_code=200):
        self._chunks=chunks
        self.status_code=status_code

    def iter_content(self, chunk_size):
        yield from self._chunks

    def raise_for_status(self):
        if self.status_code >= 400:
            raise Exception(f"{self.status_code} error")

In [84]:
fake = FakeResponse(chunks=[b"test", b"response"])

In [85]:
print(fake.status_code)
print(list(fake.iter_content(chunk_size=8192)))

200
[b'test', b'response']


In [83]:
with tempfile.TemporaryDirectory() as tmp:
    tmp_path=Path(tmp)
    fake_response=FakeResponse(chunks=[b"test,test\n",b"response,download\n"])

    with patch("src.ingestion.download.requests.get",return_value=fake_response):
        result_path=downloader(
            url="www.justatest.com", 
            dest_dir=tmp_path, 
            run_date=date(2026, 7, 8),
        )
    print(result_path)                 
    print(result_path.exists())
    print(result_path.read_bytes())

/tmp/tmpap0kdfy8/2026-07-08/products.csv.gz
True
b'test,test\nresponse,download\n'


In [87]:
with tempfile.TemporaryDirectory() as tmp:
    tmp_path=Path(tmp)
    fake_response=FakeResponse(chunks=[],status_code=404)

    with patch("src.ingestion.download.requests.get",return_value=fake_response):
        result_path=downloader(
            url="www.testingstatuscode.com",
            dest_dir=tmp_path,
            run_date=date(2026, 7, 8)
        )
print(result_path)

Exception: 404 error

In [98]:
with tempfile.TemporaryDirectory() as tmp:
    tmp_path=Path(tmp)
    fake_response=FakeResponse(chunks=[],status_code=200)

    with patch("src.ingestion.download.requests.get",return_value=fake_response):
        result_path=downloader(
            url="testdate.com",
            dest_dir=tmp_path,
        )

    print(result_path.parent.name == date.today().isoformat())

True


In [100]:
with tempfile.TemporaryDirectory() as tmp:
    tmp_path=Path(tmp)
    fake_response=FakeResponse(chunks=[],status_code=200)

    with patch("src.ingestion.download.requests.get",return_value=fake_response):
        nested_path = tmp_path / "does" / "not" / "exist" / "yet"
        result_path=downloader(
            url="testdir.com",
            dest_dir=nested_path
        )

    print(result_path)
        

/tmp/tmpm1ggy87z/does/not/exist/yet/2026-07-08/products.csv.gz


In [101]:
pwd

'/home/mark/projects/pet-food-analytics-platform/notebooks/ingestion'